Subject: ST 554 - Final Project

Name: Franklin Zhou

Date: 4/19/2026

# Fitting Your Model (50 pts)

**Create a Jupyter notebook for the modeling fitting part and the Streaming part below.**

- The file `power_ml_data.csv` is available at the URL: https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv
- You should read this data into a standard pandas data frame using the `pd.read_csv()` function.
- Convert this to a spark data frame
- We are going to treat the `Power_Zone_3` variable as our response variable.
- We can use all of the other variables as predictors. (Imagine we know that the `Power_Zone_3` reading is going to go offline in the future and we need to be able to predict that value appropriately.)

In [13]:
# Load packages and initiate spark session
import pandas as pd
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [14]:
# Read data
ml_data = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv")
df = spark.createDataFrame(ml_data) # convert to spark sql data frame
df.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

We want to fit an elastic net model using CV (no training/test split, just using CV on the data we’ve read in) with the steps below. 

The transformations below should each use an `MLlib` function that can be put into a pipeline

- The Hour column is likely not stored as a `DoubleType`. If it is not, use an SQL transformer to cast the variable as a `DoubleType`


In [15]:
# Load packages
from pyspark.ml.feature import SQLTransformer, VectorAssembler, Binarizer, OneHotEncoder, StringIndexer, PCA
from pyspark.ml import Pipeline


In [16]:
# Check the schema
df.schema

StructType([StructField('Temperature', DoubleType(), True), StructField('Humidity', DoubleType(), True), StructField('Wind_Speed', DoubleType(), True), StructField('General_Diffuse_Flows', DoubleType(), True), StructField('Diffuse_Flows', DoubleType(), True), StructField('Power_Zone_1', DoubleType(), True), StructField('Power_Zone_2', DoubleType(), True), StructField('Power_Zone_3', DoubleType(), True), StructField('Month', LongType(), True), StructField('Hour', LongType(), True)])

In [17]:
# Cast Hour to Double Type and rename Power_Zone_3 as label
cast_sql = SQLTransformer(
    statement = """
        SELECT *, CAST(Hour AS DOUBLE) AS Hour_Double
        FROM __THIS__
    """
)

- Binarize the `Hour` column based on the column being less than 6.5 or not (night vs day essentially)

In [18]:
# Binarize Hour_Double: 1 if Hour < 6.5 (night), 0 otherwise (day).
binarizer = Binarizer(
    inputCol = "Hour_Double",
    outputCol = "Hour_Bin",
    threshold = 6.5
)

- One-hot encode the Month column

In [19]:
# Cast Month to string first via a SQLTransformer, then index and encode.
cast_month_sql = SQLTransformer(
    statement = "SELECT *, CAST(Month AS STRING) AS Month_str FROM __THIS__"
)

# StringIndexer maps each month a numeric index
month_indexer = StringIndexer(
    inputCol = "Month_str",
    outputCol = "Month_idx"
)

# OneHotEncoder converts numeric index to binary vector
month_encoder = OneHotEncoder(
    inputCol = "Month_idx",
    outputCol = "Month_vector"
)

- Run a PCA fit on the `Temperature`, `Humidity`, `Wind_Speed`, `General_Diffuse_Flows`, and `Diffuse_Flows` columns.

     - To do this, I first used a `VectorAssembler()` call to place these variables in a column together for use with the `PCA()` estimator.
    
     - Once fitted, then you’ll have a PCA transformer we’ll use in our pipeline.
    
    - We’ll use two PCs in our transformation.

In [20]:
pca_assembler = VectorAssembler(
    inputCols = ["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"], 
    outputCol = "PCA_input"
)

pca = PCA(
    k = 2, 
    inputCol = "PCA_input", 
    outputCol = "PCA_features"
)

- Rename your response variable as `label`

In [21]:
# Rename Power_Zone_3 to label
label_sql = SQLTransformer(
    statement = "SELECT *, Power_Zone_3 AS label FROM __THIS__"
)

- Use VectorAssembler() to put your predictors into a features. Use the
     - two fitted PCA features
     - binary `Hour` variable
     - `Power_Zone_1`
     - `Power_Zone_2`
     - `Month` indicator variables


In [22]:
# Combine all predictors into the 'features' vector column.
features_assembler = VectorAssembler(
    inputCols = ["PCA_features", "Hour_Bin", "Power_Zone_1", "Power_Zone_2", "Month_vector"],
    outputCol = "features"
)

- This ends the pipeline of transformations!

- Now you’ll then use the `CrossValidator()` function and the LinearRegression() function to fit an elastic net model.

    - You should do the following grid for the regParam and elasticNetParam: All combinations of
    
        - regParam: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1
        
        - elasticNetParam: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1
        
- Now fit the model using 5-fold CV with `rmse` as your criterion!
        

In [23]:
# Load packages
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

In [24]:
# Setup LinearRegression instance
lr = LinearRegression()

# Setup parameters grid
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

# Setup pipeline
transformation_pipeline = Pipeline(stages=[cast_sql, binarizer, cast_month_sql, month_indexer, month_encoder, pca_assembler, pca, label_sql, features_assembler, lr])

# Create cross validation instance
crossval_lr = CrossValidator(estimator = transformation_pipeline,
                          estimatorParamMaps = paramGrid,
                          evaluator = RegressionEvaluator(metricName = 'rmse'),
                          numFolds = 5)

In [25]:
# Fit the cv model
cv_model = crossval_lr.fit(df)

26/04/19 20:48:57 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/19 20:48:57 WARN Instrumentation: [9a5f6289] regParam is zero, which might cause numerical instability and overfitting.
26/04/19 20:49:00 WARN Instrumentation: [e73fb814] regParam is zero, which might cause numerical instability and overfitting.
26/04/19 20:49:02 WARN Instrumentation: [ae0e835c] regParam is zero, which might cause numerical instability and overfitting.
26/04/19 20:49:03 WARN Instrumentation: [6a218cae] regParam is zero, which might cause numerical instability and overfitting.
26/04/19 20:49:05 WARN Instrumentation: [9bb6b14b] regParam is zero, which might cause numerical instability and overfitting.
26/04/19 20:49:06 WARN Instrumentation: [a8e9796c] regParam is zero, which might cause numerical instability and overfitting.
26/04/19 20:49:08 WARN Instrumentation: [25a78f41] regP

- Report the optimal values chosen for the tuning parameters

- Report the CV error

In [26]:
# Create a list contains RMSE value associate with parameters value
my_list = []
for i in range(len(paramGrid)):
    my_list.append([cv_model.avgMetrics[i], paramGrid[i].values()])

import numpy as np
# Convert to numpy array 
arrange = np.array(my_list)
# Sort by RMSE value
my_list_sorted = arrange[arrange[:, 0].argsort()]

# Print top 5 rows
print(my_list_sorted[:5])

[[2147.6805955040063 dict_values([0.25, 0.9])]
 [2147.6806262880464 dict_values([0.25, 0.95])]
 [2147.680652374175 dict_values([0.75, 0.25])]
 [2147.6806816732246 dict_values([0.25, 0.99])]
 [2147.680687594257 dict_values([0.25, 1.0])]]


From the output we find that the minumum RMSE is 2147.6805955040063 while the best `regParam` value is 0.25 and the best `elasticNetParam` value is 0.9.

In [27]:
# Another way to extract the value
best_lr = cv_model.bestModel.stages[-1]
# Retrieve optimal tuning parameters
print("Best regParam value is:", best_lr._java_obj.getRegParam())
print("Best elasticNetParam value is:", best_lr._java_obj.getElasticNetParam())
# CV RMSE 
print("CV RMSE:", min(cv_model.avgMetrics))

Best regParam value is: 0.25
Best elasticNetParam value is: 0.9
CV RMSE: 2147.6805955040063


- Report the training set RMSE (as done in the notes) by using your fitted model as a transformer and evaluating on the entire training set

In [28]:
# Now cv_model is a transformer with "best model" as default.
train_predictions = cv_model.transform(df)
training_rmse = RegressionEvaluator(metricName = "rmse").evaluate(train_predictions)
print("The training RMSE value is:", training_rmse)

The training RMSE value is: 2147.0975198849655


- Take the outputted transformations from the model (the predictions) and create a `residual` column (`label` - `prediction`). The `.withColumn()` method is handy here. Print out a data frame with these `residual`s, the `label` column, and the `prediction`s

In [29]:
from pyspark.sql.functions import col
# create column residual = label - prediction
res_df = train_predictions.withColumn("residual", col("label") - col("prediction")) \
             .select("label","prediction","residual")

res_df.show(10)

+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386|20879.059021624707|-638.0951616247075|
|20131.08434|18658.584292356452| 1472.500047643549|
|19668.43373|18203.134666224345|1465.2990637756557|
|18899.27711| 17589.12367656089|1310.1534334391108|
|18442.40964| 16995.81374405468|  1446.59589594532|
|18130.12048|16516.251987204687|1613.8684927953145|
|17945.06024|16091.866359951036|1853.1938800489625|
|17459.27711|15721.348901258327|1737.9282087416723|
|17025.54217|15269.749670036235|1755.7924999637653|
|16794.21687|  14937.0805978629|1857.1362721371006|
+-----------+------------------+------------------+
only showing top 10 rows


## Streaming Part (40 pts)

There is another file available at: https://www4.stat.ncsu.edu/~online/datasets/power_streaming_data.csv

Download this file and store it where your `.py` file you’ll create can find it. We’ll be randomly sampling rows from this to output to `.csv` files that you’ll be reading in.

### Reading a Stream

- We’re going to read in a stream in the form of `.csv` files. Create a folder where you will be sending your `.csv` files.
- Setup the schema for the stream (you can use the schema from the original data as we did in hw 10)
- Set up the `readStream`. Be sure to add `header = True` as you’ll likely be outputting files with a header and we don’t need to read that in.

In [30]:
data_schema = df.schema

In [31]:
# read stream data from Final_Project/stream_folder folder
stream_df = spark.readStream.option("header", True).schema(data_schema).csv("stream_folder")

### Transform/Aggregation Step

- Now, we’ll do two separate things on the stream and join them together:
    - With your stream, use your model transformer to obtain predictions from the incoming data. On the resulting predictions also create a `residual` column as noted in the previous section (return only the `label`, `prediction` and `residual` columns from this part)
    - We can use our stream more than once! With another transformation on the (original) stream, modify the response variable to be called `label`.
    - Now join your above transform with this stream based on the `label` variable which should be common to both!
    - Note 1: This is a little silly, but I want you to join two transformations of the stream and I don’t want things to get too crazy
    - Note 2: Each data frame is created from the same stream of data! You don’t need two streams, you can use the same stream and just do two separate transformations on it, combining it with a `.join()` method from one of the SQL style data frames you are dealing with (as we discussed in the notes)

In [35]:
# use model transformer to obtain predictions from the incoming data
stream_predictions = cv_model.transform(stream_df)

# Add residual column and keep only the three required columns
stream_with_residuals = stream_predictions.withColumn("residual", col("label") - col("prediction")).select("label", "prediction", "residual")

In [ ]:
# Transformation 2: Rename the response variable
stream_label_only = 

# Inner join the two stream transformations by label
joined_stream = stream_with_residuals.join(stream_label_only, on = "label", how = "inner")

### Writing Step

- Now write your stream to the console using the append output mode.

- Start the query!

In [ ]:
# Start the query
stream_query = (
    joined_stream
    .writeStream
    .outputMode("append")
    .format("console")
    .start()
)

In [ ]:
# Stop the query
stream_query.stop()

# Reference

https://share.google/aimode/nADjDIlL2cBxKdAkP
